# Batch sharing: who reuses what

Eight qubits, four independent pairs, four observables per pair
($Z_a$, $Z_b$, $Z_aZ_b$, $X_aX_b$): 16 requests. lazyqc compares full,
separate, grouped, and auto strategies; quimb contracts the same batch.
Watch evolutions, state width, and time move in different directions.

In [1]:
import os

for key in ("QULACS_NUM_THREADS", "OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ[key] = "1"

import math
import sys
import time
from pathlib import Path
from statistics import median

ROOT = Path().resolve()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "benchmarks"))

from lazyqc import Program, plan_all
from tensor_baseline import prepare as prepare_tensor


def timed(fn, repeats=5):
    values, stamps = None, []
    for _ in range(repeats):
        start = time.perf_counter_ns()
        values = fn()
        stamps.append((time.perf_counter_ns() - start) / 1e6)
    return values, median(stamps)


pairs = 4
program = Program(2 * pairs)
for j in range(pairs):
    a, b = 2 * j, 2 * j + 1
    theta = (j + 1) * math.pi / (2 * pairs + 1)
    program = program.ry(a, theta).cx(a, b)
requests = []
for j in range(pairs):
    a, b = 2 * j, 2 * j + 1
    requests += [
        program.expectation({a: "Z"}),
        program.expectation({b: "Z"}),
        program.expectation({a: "Z", b: "Z"}),
        program.expectation({a: "X", b: "X"}),
    ]
print(f"{program.n_qubits} qubits,", f"{len(requests)} requests")

8 qubits, 16 requests


## lazyqc: four strategies, one batch

- **full**: one 8-qubit state reused for everything;
- **separate**: one 2-qubit evolution per observable, no reuse;
- **grouped**: identical retained circuits shared (4 preparations);
- **auto**: heuristic pick among the above.

In [2]:
for strategy in ("full", "separate", "grouped", "auto"):
    result, ms = timed(lambda s=strategy: plan_all(requests, strategy=s).collect())
    print(
        f"{strategy:>8}: {ms:6.3f} ms | evolutions={result.stats.evolutions:>2} | "
        f"width={result.stats.max_state_qubits} | payload={result.stats.estimated_state_bytes} B"
    )

    full:  0.236 ms | evolutions= 1 | width=8 | payload=4096 B
separate:  0.357 ms | evolutions=16 | width=2 | payload=64 B
 grouped:  0.297 ms | evolutions= 4 | width=2 | payload=64 B
    auto:  0.275 ms | evolutions= 4 | width=2 | payload=64 B


## quimb on the same batch

Same `requests` objects straight into the tensor path — the middleware
framing again: no reimplementation, only a different executor.

In [3]:
tensor_values, quimb_ms = timed(lambda: prepare_tensor(requests).collect())
print(f"quimb: {quimb_ms:.3f} ms")

grouped_values = plan_all(requests, strategy="grouped").collect().values
print(
    "max |grouped - quimb|:",
    max(abs(a - b) for a, b in zip(grouped_values, tensor_values)),
)

quimb: 28.291 ms
max |grouped - quimb|: 1.2212453270876722e-15


## Reading the trade

- Width reduction alone does not predict time: separate cones run 16
  tiny evolutions and usually lose to fewer shared ones.
- Grouping wins by avoiding repetition (4 shared runs instead of 16),
  with a 64-byte largest state against full's 4096 bytes.
- quimb factorizes this family too — its answer matches, but every call
  re-searches contraction paths and nothing caps state payload; those
  are planner features, not physics.
- Honest miss to carry: at 2–8 qubits dense execution wins outright and
  auto can misselect grouping. The crossover for this family sits
  between 8 and 12 qubits — workload-specific, never a hard threshold.